# 12 · Interseismic Coupling

*Use backslip to image the locked fraction of a fault from surface velocity.*

**Learning objectives**

- derive the locked screw-dislocation velocity profile
- state GeoDef's reference-frame and backslip sign convention
- estimate bounded plate-parallel backslip with velocity metadata
- convert backslip rate to coupling and moment-deficit rate

**Prerequisites:** Chapters 05 and 07  
**Estimated time:** 45–60 minutes


## Motivation

Between earthquakes, GNSS stations move steadily as plates converge while the
shallow fault remains locked. Elastic strain accumulates around that locked
zone. Backslip is a kinematic construction that represents the missing steady
slip with an opposite-sense dislocation, turning coupling into the same linear
inverse problem used for coseismic slip.


## The arctangent profile

For an infinitely long vertical strike-slip fault locked to depth $D$ and
moving at far-field relative rate $V$, the surface-parallel velocity is

$$v(x)=\frac{V}{\pi}\arctan\left(\frac{x}{D}\right).$$

The transition broadens with locking depth. Near-fault gradient therefore
constrains $D$ more directly than far-field velocity does.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

x_km = np.linspace(-100.0, 100.0, 401)
plate_rate_mm_yr = 40.0
fig, ax = plt.subplots(figsize=(7, 3.5), constrained_layout=True)
for locking_depth_km in (5.0, 15.0, 30.0):
    velocity = plate_rate_mm_yr / np.pi * np.arctan(x_km / locking_depth_km)
    ax.plot(x_km, velocity, label=f"D = {locking_depth_km:g} km")
ax.set(xlabel="Fault-normal distance (km)", ylabel="Fault-parallel velocity (mm/yr)", title="Locked screw-dislocation profile")
ax.legend();


## Reference frame and backslip convention

Correct observed velocities for rigid-block motion first, normally into an
overriding-plate-fixed frame. Use the same Euler-pole vocabulary for station
correction and patch plate-motion directions; pole uncertainty becomes data
uncertainty.

Positive backslip amplitude is anti-parallel to local plate motion. On a
megathrust it is normal-sense motion, so its raw dip-slip component is negative.
The plate basis hides this bookkeeping: point `plate_rake` in the backslip
direction and bound its parallel amplitude between zero and local plate rate.
The binding sign mapping is recorded in
[conventions](../docs/conventions.md#interseismic-backslip-and-coupling).


## A small coupling inversion

The synthetic velocities below are already in the overriding-plate-fixed
frame. The declared convergence rate is 50 mm/yr; coupling varies from 0 to 1.
This teaching example uses a uniform local backslip direction. A real curved
mesh would obtain per-patch direction from `slip.plate_rake_from_euler`.


In [ ]:
rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=12_000.0, strike=90.0, dip=20.0,
    length=40_000.0, width=24_000.0, n_length=5, n_width=3,
)
plate_rate = 0.05  # m/yr
plate_rake = -90.0  # normal-sense, anti-parallel backslip on this thrust
i = np.arange(fault.n_patches) % 5
j = np.arange(fault.n_patches) // 5
coupling_true = np.exp(-((i - 2.0) / 1.6) ** 2 - ((j - 0.7) / 1.0) ** 2)
backslip_true = plate_rate * coupling_true
strike_rate, dip_rate = geodef.slip.from_rake(backslip_true, rake_degrees=plate_rake)

station_lon, station_lat = np.meshgrid(
    np.linspace(-118.30, -117.70, 8), np.linspace(33.82, 34.22, 6)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=strike_rate, slip_dip=dip_rate
)
sigma = 0.0008  # m/yr
velocity = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    quantity="velocity", units="m/yr", name="block_corrected_gnss",
)
result = geodef.solve(
    fault, datasets=velocity, components="plate", plate_rake=plate_rake,
    regularization="laplacian", regularization_strength=1.0,
    bounds=(np.array([0.0, -np.inf]), np.array([plate_rate, np.inf])),
)
coupling = result.rake_parallel / plate_rate
moment_deficit_rate = fault.medium.mu * np.sum(
    fault.areas * result.rake_parallel
)
print(f"moment-deficit rate: {moment_deficit_rate:.3e} N m/yr")
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
geodef.plot.patches(fault, coupling_true, ax=axes[0], vmin=0, vmax=1, title="Input coupling", colorbar_label="Coupling")
geodef.plot.patches(fault, coupling, ax=axes[1], vmin=0, vmax=1, title="Recovered coupling", colorbar_label="Coupling");


The plate-parallel solution is a backslip rate, not a signed
coseismic slip. Dividing by the local plate rate produces a fraction. Formal
uncertainty still excludes Euler-pole error, block-model error, and viscoelastic
or transient deformation.


## Checkpoint questions

**Why correct the reference frame before coupling inversion?**
<details><summary>Answer</summary>Rigid motion otherwise masquerades as elastic
strain or coupling.</details>

**Why is positive backslip raw dip slip negative here?**
<details><summary>Answer</summary>It is normal-sense motion anti-parallel to
thrust convergence.</details>

**Is backslip a friction law?**
<details><summary>Answer</summary>No. It is a kinematic superposition device.</details>


## Common mistakes

- Inverting uncorrected velocities mixes block rotation with elastic loading.
- Dividing by one scalar rate on a curved boundary can misstate coupling.
- Plotting signed dip slip instead of coupling obscures the physical fraction.
- Treating moment-deficit rate as an earthquake forecast ignores release timing.


## Recap

- Locking is represented by opposite-sense backslip in a declared plate basis.
- Coupling is bounded backslip rate divided by local plate rate.
- Reference-frame and pole uncertainty are part of the inference.

Use the plate-basis and constraints guides in [workflow](../docs/workflow.md).


## Exercises

Worked solutions are in `solutions/12_interseismic_coupling_solutions.ipynb`.

1. Fit arctangent profiles for several locking depths.
2. Move the coupled patch down dip and compare recoverability.
3. Perturb plate rate by 10% and quantify coupling and moment-deficit bias.
4. Challenge: express the synthetic problem in raw dip-slip coordinates and
   verify the plate-basis prediction exactly.


## Further reading

- Savage (1983), the backslip model of interseismic deformation.
- Segall (2010), earthquake-cycle geodesy and coupling.
- McCaffrey (2002), block models and GPS velocities.
